# Prototyping the Lead Data Reshape
Building the lead cleaning step one stage at a time before moving it into `transform.py`. The goal: go from one row per neighborhood (with a column for every year) to one row per neighborhood per year.

In [208]:
import pandas as pd

## Stage 1: Load, filter, and select columns

In [209]:
lead = pd.read_csv("../data/raw/blood_lead.csv")
lead = lead[lead["community_area_number"].between(1, 77)]

In [210]:
screened_cols = [f"screened_for_lead_in_{year}" for year in range(1999, 2014)]
elevated_cols = [f"percent_elevated_{year}" for year in range(1999, 2014)]
keep_cols = ["community_area_number"] + screened_cols + elevated_cols
len(keep_cols)

31

In [211]:
lead = lead[keep_cols]
lead.shape

(77, 31)

77 neighborhoods, 1 ID column + 30 year columns.

## Stage 2: Melt from wide to long

In [212]:
long = lead.melt(id_vars=["community_area_number"], var_name="column", value_name="value")
long.shape

(2310, 3)

2,310 rows = 77 neighborhoods x 30 columns, so nothing was lost or duplicated.

In [213]:
long.head(20)

,community_area_number,column,value
0,1,screened_for_lead_in_1999,2449.0
1,2,screened_for_lead_in_1999,1624.0
2,3,screened_for_lead_in_1999,2247.0
3,4,screened_for_lead_in_1999,1027.0
4,5,screened_for_lead_in_1999,516.0
5,6,screened_for_lead_in_1999,632.0
6,7,screened_for_lead_in_1999,364.0
7,8,screened_for_lead_in_1999,1353.0
8,9,screened_for_lead_in_1999,67.0
9,10,screened_for_lead_in_1999,248.0


## Stage 3: Split year and measure out of the column name

Every name ends with an underscore and a 4-digit year, so slicing from the end works for both measures.

In [214]:
long["year"] = long["column"].str[-4:].astype(int)
long["measure"] = long["column"].str[:-5]

In [215]:
long["measure"].unique()

<StringArray>
['screened_for_lead_in', 'percent_elevated']
Length: 2, dtype: str

In [216]:
long["year"].min(), long["year"].max()

(np.int64(1999), np.int64(2013))

## Stage 4: Pivot measures back into columns

One row per neighborhood-year, with each measure in its own column.

In [217]:
lead_long = long.pivot(index=["community_area_number", "year"], columns="measure", values="value").reset_index()
lead_long = lead_long.rename(columns={"screened_for_lead_in": "kids_screened"})
lead_long.columns.name = None
lead_long.shape

(1155, 4)

1,155 rows = 77 neighborhoods x 15 years.

In [218]:
lead_long.head()

,community_area_number,year,percent_elevated,kids_screened
0,1,1999,10.5,2449.0
1,1,2000,7.7,2364.0
2,1,2001,5.8,2501.0
3,1,2002,4.8,2219.0
4,1,2003,4.4,2182.0


Spot-check one value against the original wide data.

In [219]:
lead[lead["community_area_number"] == 1]["percent_elevated_1999"]

1    10.5
Name: percent_elevated_1999, dtype: float64

## Stage 5: Zero screenings are missing data

Checking all years at once.

In [220]:
lead_long[lead_long["kids_screened"] == 0]

,community_area_number,year,percent_elevated,kids_screened
799,54,2003,0.0,0.0
800,54,2004,0.0,0.0
801,54,2005,0.0,0.0
802,54,2006,0.0,0.0
803,54,2007,0.0,0.0
804,54,2008,0.0,0.0
805,54,2009,0.0,0.0
806,54,2010,0.0,0.0
807,54,2011,0.0,0.0
808,54,2012,0.0,0.0


Riverdale (#54) has zero screenings for every year from 2003-2013, not just the two years found in exploration. Its lead data only exists for 1999-2002. Setting these to missing so they aren't counted as real zeros.

In [221]:
zero_screened = lead_long["kids_screened"] == 0
lead_long.loc[zero_screened, ["kids_screened", "percent_elevated"]] = pd.NA
lead_long["kids_screened"] = lead_long["kids_screened"].astype("Int64")

In [222]:
lead_long.isna().sum()

community_area_number     0
year                      0
percent_elevated         11
kids_screened            11
dtype: int64

In [223]:
lead_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 1155 entries, 0 to 1154
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   community_area_number  1155 non-null   int64  
 1   year                   1155 non-null   int64  
 2   percent_elevated       1144 non-null   float64
 3   kids_screened          1144 non-null   Int64  
dtypes: Int64(1), float64(1), int64(2)
memory usage: 37.4 KB


11 missing in each measure (Riverdale 2003-2013). year is an integer to match the crime data, and kids_screened uses Int64 so it can hold missing values.

## Result
This logic is now `clean_lead()` in `src/transform.py`.

Two things the prototype caught that a script alone might have missed:
- Riverdale is missing 11 years of data, not 2.
- `year` came out as text after slicing. `min()` and `max()` looked right, but `info()` showed the wrong type.